# DSPy structured extraction — tweak the prompt from (text, dict) pairs

Give DSPy examples of `(text, extracted_dict)` and it optimizes the extraction
**prompt** (instruction + few-shot demos) to match. Uses the OpenAI LM with
DSPy's **JSON adapter**, which formats/parses via OpenAI structured outputs —
the right adapter for extracting a dictionary.

`pip install dspy` · `export OPENAI_API_KEY=...`

In [ ]:
import os
from types import SimpleNamespace
import dspy

lm = dspy.LM('openai/gpt-4o-mini', api_key=os.getenv(''))
# local vLLM instead: dspy.LM('openai/local-model', api_base='http://localhost:8000/v1', api_key='EMPTY')
dspy.configure(lm=lm, adapter=dspy.JSONAdapter())   # JSON adapter -> OpenAI structured output

## Data — toy pairs or your own DataFrame

Each field maps to a **list of labels** drawn from `ALLOWED` (the possible labels
per field); `issue` is a list of `[product, issue]` pairs. `ALLOWED` is injected
into the prompt so the model only picks from those labels.

Default: the built-in toy set. Set `USE_DATAFRAME = True` to build examples from a
DataFrame (pick the text column + field columns), and set `ALLOWED` to your vocab.

In [28]:
import pandas as pd


def to_examples(df, text_col, field_cols):
    """DataFrame -> DSPy examples: text_col is the input, field_cols become the dict to extract."""
    return [dspy.Example(text=r[text_col],
                         extracted={c: r[c] for c in field_cols}).with_inputs('text')
            for _, r in df.iterrows()]


# allowed label vocabulary per field — the agent is told to use ONLY these
ALLOWED = {
    'product': ['blender', 'kitchen equipment', 'headphones', 'audio', 'laptop', 'computer',
                'coffee maker', 'monitor', 'chair', 'furniture', 'keyboard', 'phone case',
                'shoes', 'printer', 'speaker', 'watch', 'wearable'],
    'sentiment': ['positive', 'negative', 'neutral', 'complaint', 'praise'],
    'issue': ['stopped working', 'slow shipping', 'leaks', 'scratched', 'cracked',
              'paper jams', 'short battery life'],
}

USE_DATAFRAME = False   # flip to True to extract from your own DataFrame (set ALLOWED to your vocab)

if USE_DATAFRAME:
    df = pd.read_csv('your_data.csv')                 # must contain the columns below
    TEXT_COL = 'text'
    FIELD_COLS = ['product', 'sentiment', 'issue']    # columns -> extracted dict keys
    examples = to_examples(df, TEXT_COL, FIELD_COLS)
else:
    # toy dataset: each field is a LIST of labels; 'issue' is a list of [product, issue] pairs
    DATA = [
        ('The blender stopped working after two days.', {'product': ['blender', 'kitchen equipment'], 'sentiment': ['negative', 'complaint'], 'issue': [['blender', 'stopped working']]}),
        ('Absolutely love these headphones, crystal clear sound.', {'product': ['headphones', 'audio'], 'sentiment': ['positive', 'praise'], 'issue': []}),
        ('The laptop is fine but shipping took three weeks.', {'product': ['laptop', 'computer'], 'sentiment': ['neutral'], 'issue': [['laptop', 'slow shipping']]}),
        ('This coffee maker leaks all over the counter.', {'product': ['coffee maker', 'kitchen equipment'], 'sentiment': ['negative', 'complaint'], 'issue': [['coffee maker', 'leaks']]}),
        ('Great value monitor, sharp and bright.', {'product': ['monitor', 'computer'], 'sentiment': ['positive', 'praise'], 'issue': []}),
        ('The chair arrived with a scratched leg.', {'product': ['chair', 'furniture'], 'sentiment': ['negative', 'complaint'], 'issue': [['chair', 'scratched']]}),
        ('Keyboard works as described, nothing special.', {'product': ['keyboard', 'computer'], 'sentiment': ['neutral'], 'issue': []}),
        ('My phone case cracked within a week.', {'product': ['phone case'], 'sentiment': ['negative', 'complaint'], 'issue': [['phone case', 'cracked']]}),
        ('These shoes are comfortable and stylish.', {'product': ['shoes'], 'sentiment': ['positive', 'praise'], 'issue': []}),
        ('The printer keeps jamming on every third page.', {'product': ['printer', 'computer'], 'sentiment': ['negative', 'complaint'], 'issue': [['printer', 'paper jams']]}),
        ('Speaker sounds decent for the price.', {'product': ['speaker', 'audio'], 'sentiment': ['neutral'], 'issue': []}),
        ('Battery on this watch dies after a few hours.', {'product': ['watch', 'wearable'], 'sentiment': ['negative', 'complaint'], 'issue': [['watch', 'short battery life']]}),
    ]
    examples = [dspy.Example(text=t, extracted=d).with_inputs('text') for t, d in DATA]

k = int(len(examples) * 0.7)                # 70/30 split, works for any size
trainset, testset = examples[:k], examples[k:]
print(len(trainset), 'train /', len(testset), 'test')

8 train / 4 test


## Program + metric

In [29]:
from pydantic import BaseModel


class Extraction(BaseModel):
    """Structured target: lists of labels; 'issue' is a list of [product, issue] pairs.

    Typed fields (not a bare dict) so the JSON adapter emits a real schema — a plain
    `dict` output makes OpenAI structured outputs return {}.
    """
    product: list[str] = []
    sentiment: list[str] = []
    issue: list[list[str]] = []   # each item: [product, issue]


class Extract(dspy.Signature):
    """Extract structured fields from the product comment.

    Use ONLY labels from the allowed set for each field; if a field does not apply, return []."""
    text: str = dspy.InputField()
    extracted: Extraction = dspy.OutputField()   # 'extracted' avoids clashing with Signature.fields


# tell the agent the allowed vocabulary so it only uses those labels
Extract = Extract.with_instructions(Extract.instructions + f"\n\nAllowed labels per field:\n{ALLOWED}")

extract = dspy.Predict(Extract)


def _as_dict(x):
    """Pydantic model or dict -> dict."""
    if x is None:
        return {}
    return x.model_dump() if hasattr(x, 'model_dump') else x


def _to_set(v):
    """Field value (list, list-of-pairs, or scalar) -> a set of comparable items."""
    norm = lambda x: (tuple(str(e).strip().lower() for e in x)
                      if isinstance(x, (list, tuple)) else str(x).strip().lower())
    if v is None:
        return set()
    return {norm(x) for x in (v if isinstance(v, (list, tuple)) else [v])}


def _f1(gold, got):
    """Set F1 of one field's predicted labels vs gold labels."""
    g, p = _to_set(gold), _to_set(got)
    if not g and not p:
        return 1.0
    if not g or not p:
        return 0.0
    tp = len(g & p)
    prec, rec = tp / len(p), tp / len(g)
    return 2 * prec * rec / (prec + rec) if prec + rec else 0.0


def f1_macro(example, pred, trace=None):
    """Macro F1: per-field set F1, averaged equally across fields."""
    gold = example.extracted
    got = _as_dict(getattr(pred, 'extracted', {}))
    keys = set(gold) | set(got)
    if not keys:
        return 1.0
    return sum(_f1(gold.get(k), got.get(k)) for k in keys) / len(keys)


def _check_metric():
    e = dspy.Example(text='x', extracted={'product': ['a', 'b'], 'issue': [['a', 'x']]})
    assert f1_macro(e, SimpleNamespace(extracted={'product': ['a', 'b'], 'issue': [['a', 'x']]})) == 1.0
    # product {a,b} vs {a}: F1=2/3 ; issue {(a,x)} vs {}: 0 ; macro = 1/3
    assert abs(f1_macro(e, SimpleNamespace(extracted={'product': ['a'], 'issue': []})) - 1 / 3) < 1e-9
    print('metric ok')


_check_metric()

metric ok


In [30]:
def run_eval(program, dataset, title):
    """Run the extractor on each test item and print gold vs predicted dict."""
    print(f'\n=== {title} ===')
    scores = []
    for ex in dataset:
        pred = program(text=ex.text)
        got = _as_dict(getattr(pred, 'extracted', {}))
        s = f1_macro(ex, pred)
        scores.append(s)
        print(f'  {s:.2f}  {ex.text[:55]}')
        print(f'        gold={ex.extracted}')
        print(f'        pred={got}')
    print(f'  macro F1: {sum(scores) / len(scores):.3f}')
    return sum(scores) / len(scores)

## Baseline → optimize the prompt → after

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = ''    
run_eval(extract, testset, 'baseline (unoptimized prompt)')


=== baseline (unoptimized prompt) ===
  0.89  These shoes are comfortable and stylish.
        gold={'product': ['shoes'], 'sentiment': ['positive', 'praise'], 'issue': []}
        pred={'product': ['shoes'], 'sentiment': ['positive'], 'issue': []}
  0.78  The printer keeps jamming on every third page.
        gold={'product': ['printer', 'computer'], 'sentiment': ['negative', 'complaint'], 'issue': [['printer', 'paper jams']]}
        pred={'product': ['printer'], 'sentiment': ['complaint'], 'issue': [['printer', 'paper jams']]}
  0.56  Speaker sounds decent for the price.
        gold={'product': ['speaker', 'audio'], 'sentiment': ['neutral'], 'issue': []}
        pred={'product': ['speaker'], 'sentiment': ['positive'], 'issue': []}
  0.78  Battery on this watch dies after a few hours.
        gold={'product': ['watch', 'wearable'], 'sentiment': ['negative', 'complaint'], 'issue': [['watch', 'short battery life']]}
        pred={'product': ['watch'], 'sentiment': ['complaint'], 'iss

0.7499999999999999

In [32]:
from dspy.teleprompt import MIPROv2

# MIPROv2 rewrites the instruction and picks few-shot demos to maximize macro F1
optimizer =MIPROv2(metric=f1_macro, auto='light')
optimized = optimizer.compile(extract, trainset=trainset, requires_permission_to_run=False)
#dspy.teleprompt.BootstrapFewShot(metric=f1_macro).compile(extract, trainset=trainset)
#optimized=dspy.teleprompt.BootstrapFewShot(metric=f1_macro).compile(extract, trainset=trainset)
run_eval(optimized, testset, 'optimized prompt')

2026/07/31 14:37:41 WARNING dspy.teleprompt.mipro_optimizer_v2: 'requires_permission_to_run' is deprecated and will be removed in a future version.
2026/07/31 14:37:41 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 6

2026/07/31 14:37:41 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/07/31 14:37:41 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/07/31 14:37:41 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 2/2 [00:01<00:00,  1.08it/s]


Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 4/6


100%|██████████| 2/2 [00:00<00:00, 13.17it/s]


...


Average Metric: 4.78 / 6 (79.6%): 100%|██████████| 6/6 [00:00<00:00, 36.18it/s]

2026/07/31 14:38:07 INFO dspy.evaluate.evaluate: Average Metric: 4.777777777777777 / 6 (79.6%)


2026/07/31 14:38:07 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 79.63 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0'].
2026/07/31 14:38:07 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [79.63, 57.41, 57.41, 55.56, 50.0, 68.52, 57.41, 50.0, 55.56, 50.0, 79.63]
2026/07/31 14:38:07 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 79.63
2026/07/31 14:38:07 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/07/31 14:38:07 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 79.63!



=== optimized prompt ===
  0.89  These shoes are comfortable and stylish.
        gold={'product': ['shoes'], 'sentiment': ['positive', 'praise'], 'issue': []}
        pred={'product': ['shoes'], 'sentiment': ['positive'], 'issue': []}
  0.78  The printer keeps jamming on every third page.
        gold={'product': ['printer', 'computer'], 'sentiment': ['negative', 'complaint'], 'issue': [['printer', 'paper jams']]}
        pred={'product': ['printer'], 'sentiment': ['complaint'], 'issue': [['printer', 'paper jams']]}
  0.56  Speaker sounds decent for the price.
        gold={'product': ['speaker', 'audio'], 'sentiment': ['neutral'], 'issue': []}
        pred={'product': ['speaker'], 'sentiment': ['positive'], 'issue': []}
  0.78  Battery on this watch dies after a few hours.
        gold={'product': ['watch', 'wearable'], 'sentiment': ['negative', 'complaint'], 'issue': [['watch', 'short battery life']]}
        pred={'product': ['watch'], 'sentiment': ['complaint'], 'issue': [['watch

0.7499999999999999

## See the tweaked prompt

In [ ]:
print('=== optimized instruction ===')
print(optimized.signature.instructions)
print(f'\n=== few-shot demos: {len(optimized.demos)} ===')
for d in optimized.demos:
    print(' ', {k: d.get(k) for k in ('text', 'extracted')})

dspy.inspect_history(n=1)   # the actual prompt+response of the last call
optimized.save('extract_optimized.json')